# Pandas Construction

Companion notebook for Tutorial 09. Build a `surge.Network` directly from
pandas DataFrames — no intermediate file needed.

## Minimal 3-Bus Network

Supply buses, branches, and generators as DataFrames. Only a handful of
columns are required; everything else has sensible defaults.

In [ ]:
import pandas as pd
import surge

buses = pd.DataFrame(
    [
        {"number": 1, "type": "Slack", "base_kv": 230.0, "name": "SWING"},
        {"number": 2, "type": "PV", "base_kv": 230.0, "name": "GEN"},
        {"number": 3, "type": "PQ", "base_kv": 230.0, "name": "LOAD", "pd_mw": 90.0, "qd_mvar": 30.0},
    ]
)
branches = pd.DataFrame(
    [
        {"from_bus": 1, "to_bus": 2, "r": 0.01, "x": 0.08, "rate_a": 250.0},
        {"from_bus": 2, "to_bus": 3, "r": 0.01, "x": 0.09, "rate_a": 250.0},
        {"from_bus": 1, "to_bus": 3, "r": 0.02, "x": 0.12, "rate_a": 250.0},
    ]
)
generators = pd.DataFrame(
    [
        {"bus": 1, "pg": 100.0, "pmax": 150.0, "qmax": 80.0, "qmin": -80.0},
        {"bus": 2, "pg": 40.0, "pmax": 80.0, "vs": 1.02},
    ]
)

net = surge.construction.from_dataframes(
    buses,
    branches,
    generators,
    base_mva=100.0,
    name="pandas-demo",
)

print(f"Buses: {net.n_buses}  Branches: {net.n_branches}")
result = surge.solve_ac_pf(net)
print(f"Converged: {result.converged}  Iterations: {result.iterations}")

## Run It Through a Full Workflow

The constructed network is a normal `surge.Network` — it works with
every solver and analysis tool.

In [ ]:
dc = surge.solve_dc_pf(net, surge.DcPfOptions())
print("DC bus angles (rad):", dc.va_rad)
print("DC branch flows (MW):", dc.branch_p_mw)

ac = surge.solve_ac_pf(net, surge.AcPfOptions())
print("AC voltages (pu):", ac.vm)
print("AC max mismatch:", ac.max_mismatch)

## Optional Columns

Add line charging, transformer taps, phase shifts, or parallel circuits
via optional branch columns. Extra columns are silently ignored.

In [ ]:
branches_extended = pd.DataFrame(
    [
        {"from_bus": 1, "to_bus": 2, "r": 0.01, "x": 0.08, "b": 0.02, "rate_a": 250.0},
        {"from_bus": 2, "to_bus": 3, "r": 0.01, "x": 0.09, "b": 0.01, "rate_a": 250.0},
        {"from_bus": 1, "to_bus": 3, "r": 0.0, "x": 0.10, "tap": 1.05, "rate_a": 200.0},
    ]
)

net2 = surge.construction.from_dataframes(
    buses,
    branches_extended,
    generators,
    base_mva=100.0,
    name="with-transformer",
)

ac2 = surge.solve_ac_pf(net2)
print(f"Converged: {ac2.converged}  Iterations: {ac2.iterations}")
print("Voltages:", ac2.vm)